In [14]:
#==========================================PHASE 1 : DATA CLEANING===============================================

import pandas as pd
import numpy as np

df = pd.read_csv(r"F:\Project\Spotify\data\raw\tracks.csv")


print("Dataset shape : ",df.shape)

print("Dataset info : ")
print(df.info())

print("Column Names : ")
print(df.columns.tolist())

print("Missing values : ")
print(df.isnull().sum())
print("Here some of the data's are missing, we will fill the missing values with unknown")


Dataset shape :  (899702, 16)
Dataset info : 
<class 'pandas.DataFrame'>
RangeIndex: 899702 entries, 0 to 899701
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          899702 non-null  str    
 1   genres            892516 non-null  str    
 2   track_artists     99943 non-null   str    
 3   tempo             899224 non-null  float64
 4   energy            899224 non-null  float64
 5   key               899224 non-null  float64
 6   popularity        899701 non-null  float64
 7   mode              899224 non-null  float64
 8   time_signature    899224 non-null  float64
 9   speechiness       899224 non-null  float64
 10  danceability      899224 non-null  float64
 11  valence           899224 non-null  float64
 12  acousticness      899224 non-null  float64
 13  liveness          899224 non-null  float64
 14  instrumentalness  899224 non-null  float64
 15  name              899215 non-null

In [31]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"F:\Project\Spotify\data\raw\tracks.csv")

#This creates a copy of the raw database
df_clean = df.copy()

print("Datasets shape : ",df.shape)

#Checking the duplicate values
print("Duplicate values : ",df_clean.duplicated().sum())
#Here there is no duplicate values in the dataset

#Checking the duplicate values from ttrack_id
print("Duplicate values from track_id : ",df_clean['track_id'].duplicated().sum())
#Here there is no duplicate values in the track_id column


#Checking the missing values in the dataset based on the columns
missing_percentage = (
    df_clean.isnull().sum() / len(df_clean) *100
).sort_values(ascending=False)
print(missing_percentage)


#                              STEP : 1
#Here track_artists has (88.891544) highest percentage of missing values
#So now i use fillna() method to fill the missing values with unknown because it has the highest percentage of missing values.We can't drop the column
df_clean['track_artists'] = df_clean['track_artists'].fillna('unknown')

print("After filling the missing values in track_artists column : ",df_clean['track_artists'].isnull().sum())


#                              STEP : 2
#Here genres has (0.798709) percentage of missing values
df_clean['genres']=df_clean['genres'].fillna('unknown')
print("After filling the missing values in genres column : ",df_clean['genres'].isnull().sum())


#                              STEP : 3
#Here name has (0.054129) percentage of missing values
df_clean['name']=df_clean['name'].fillna('unknown')
print("After filling the missing values in name column : ",df_clean['name'].isnull().sum())


#                              STEP : 4
#Here about 11 columns has the same percentage of missing values (0.053129).So we can drop the rows
#I am selecting all the rows and make it as a single row so we can easily drop the rows 

audio = [
    'tempo',
    'mode',
    'key',
    'time_signature',
    'energy',
    'valence',
    'acousticness',
    'danceability',
    'speechiness',
    'liveness',
    'instrumentalness'
]
missing_audio = df_clean[
    df_clean[audio].isnull().all(axis=1)
]
print("Missing audio rows : ",missing_audio.shape[0])

df_clean = df_clean.dropna(subset=audio)
print("After dropping the missing audio rows : ",df_clean.shape[0])


#                              STEP : 5
#Here popularity has (0.000111) percentage of missing values
df_clean=df_clean.dropna(subset=['popularity'])
print("Missing popularity rows : ",df_clean['popularity'].isnull().sum())



#Final Checkup
df_clean = df_clean.reset_index(drop=True)
print("Final Dataset shape : ",df_clean.shape)

print("\nFinal Missing Values:")
print(df_clean.isnull().sum())

print("\nFinal Duplicate Rows:")
print(df_clean.duplicated().sum())

print("\nFinal Duplicate Track IDs:")
print(df_clean['track_id'].duplicated().sum())

print("\nData Types:")
print(df_clean.dtypes)
df_clean.info()

print(df_clean.describe())

#Checking for invalid values

audio_features = [
    'energy',
    'speechiness',
    'danceability',
    'valence',
    'acousticness',
    'liveness',
    'instrumentalness'
]
for col in audio_features:
    invalid_values = df_clean[
        (df_clean[col]<0) | 
        (df_clean[col]>1)
    ]
    print(f"{col} : {len(invalid_values)} invalid values")


#Check the popularity
invalid_popularity = df_clean[
    (df_clean['popularity']<0) | 
    (df_clean['popularity']>100)
]
print("Invalid popularity values:", len(invalid_popularity))


#Check Tempo
invalid_tempo = df_clean[
    (df_clean['tempo']<0)
]
print("Invalid tempo values:", len(invalid_tempo))


#Check Musical Key
invalid_key = df_clean[
    (df_clean['key']<0) | 
    (df_clean['key']>11)
]
print("Invalid musical key values:", len(invalid_key))


print(df_clean['mode'].value_counts())


#Check Time Signature
print(
    df_clean['time_signature']
    .value_counts()
    .sort_index()
)


#Detect outliers in the dataset using IQR method
outlier_columns = [
    'tempo',
    'popularity',
    'energy',
    'speechiness',
    'danceability',
    'valence',
    'acousticness',
    'liveness',
    'instrumentalness'
]

def check_outliers(data,column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[
        (data[column] < lower_bound) | 
        (data[column] > upper_bound)
    ]
    return len(outliers),lower_bound,upper_bound

for col in outlier_columns:
    count, lower, upper = check_outliers(
        df_clean,
        col
    )
    print(f"\nColumn: {col}")
    print(f"Number of outliers: {count}")
    print(f"Lower Bound: {lower}")
    print(f"Upper Bound: {upper}")



#Check Whether Outliers Are Within Valid Ranges
numeric_columns = [
    'tempo',
    'popularity',
    'energy',
    'speechiness',
    'danceability',
    'valence',
    'acousticness',
    'liveness',
    'instrumentalness'
]

for col in numeric_columns:
    print(
        f"{col}: "
        f"Min = {df_clean[col].min()}, "
        f"Max = {df_clean[col].max()}"
    )

print("Tracks with tempo = 0:", (df_clean['tempo'] == 0).sum())
df_clean = df_clean[df_clean['tempo'] > 0]
df_clean = df_clean.reset_index(drop=True)
print("Final Dataset Shape:", df_clean.shape)
print("Tempo = 0 rows:", (df_clean['tempo'] == 0).sum())


print(df_clean.isnull().sum())
print("Duplicate rows:", df_clean.duplicated().sum())
print("Tempo = 0:", (df_clean['tempo'] == 0).sum())
print("Final shape:", df_clean.shape)


#Saving the cleaned dataset to a new CSV file
df_clean.to_csv(
    r"F:\Project\Spotify\data\processed\spotify_cleaned.csv",
    index=False
)
print("Cleaned dataset saved to 'spotify_cleaned.csv'")


Datasets shape :  (899702, 16)
Duplicate values :  0
Duplicate values from track_id :  0
track_artists       88.891544
genres               0.798709
name                 0.054129
tempo                0.053129
mode                 0.053129
key                  0.053129
time_signature       0.053129
energy               0.053129
valence              0.053129
acousticness         0.053129
danceability         0.053129
speechiness          0.053129
liveness             0.053129
instrumentalness     0.053129
popularity           0.000111
track_id             0.000000
dtype: float64
After filling the missing values in track_artists column :  0
After filling the missing values in genres column :  0
After filling the missing values in name column :  0
Missing audio rows :  478
After dropping the missing audio rows :  899224
Missing popularity rows :  0
Final Dataset shape :  (899223, 16)

Final Missing Values:
track_id            0
genres              0
track_artists       0
tempo             